# Visor MuJoCo: freshfile_description

## Por que no se carga el xacro directo

`urdf/freshfile_description/urdf/freshfile.xacro` no es cargable por MuJoCo: es xacro (usa `<xacro:include>` y `${...}`) y sus mallas apuntan con sintaxis ROS `file://$(find freshfile_description)/...`, que MuJoCo no resuelve. Tampoco hay `xacro` instalado para preprocesarlo.

Los datos en si (mallas STL, inercias, 4 juntas continuas) si alcanzan, asi que se genero una version aplanada en URDF puro: [`freshfile_mujoco.urdf`](../../urdf/freshfile_description/urdf/freshfile_mujoco.urdf).

## Auditoria de la conversion

Lo que se verifico contra el modelo ya compilado, no solo "carga sin error":

| Chequeo | Resultado |
|---|---|
| Escala de mallas (`scale=0.001`, mm->m) | OK - bboxes de 0.04 a 0.21 m, brazo de ~0.25 m |
| Ensamblaje cinematico | OK - los offsets de malla del exportador cancelan los offsets de frame; las piezas quedan co-locadas, no explotadas |
| Inercias | OK - aceptadas sin `balanceinertia`; MuJoCo solo las diagonaliza (los productos de inercia eran chicos) |
| Masa total | 2.364 kg = suma de los 4 links moviles |
| Estabilidad | OK - 20 s de caida libre sin NaN ni warnings |

**Dos cosas que hay que saber, y una que era un bug real:**

1. **`base_link` desaparece como body.** Es el link raiz y no tiene junta propia, asi que MuJoCo lo suelda al mundo: su geom se dibuja igual, pero su masa (1.093 kg) se descarta, porque el mundo tiene masa infinita. Es correcto para una base fija; solo no busques `base_link` en `model.body_*`.

2. **Los geoms visuales se descartan.** Para URDF, MuJoCo usa `discardvisual=true` por defecto, asi que los 5 geoms son los de colision. Aca da igual: visual y colision usan la misma malla con el mismo origen.

3. **BUG (corregido): `Revolute_1` estaba trabado.** Como `base_link` vive en el body mundo, y MuJoCo *no* aplica su filtro de colision padre-hijo cuando el padre es el mundo, `base_link` y `rot_1` chocaban. Peor: MuJoCo colisiona mallas por su **casco convexo**, que rellena el hueco del hombro donde las piezas encastran, asi que arrancaban con 13 mm de interpenetracion. El contacto permanente dejaba la junta de la base muerta:

   | Patada de 5 rad/s, 2 s de sim | `qpos[0]` | `qvel[0]` |
   |---|---|---|
   | contactos ON (antes) | +0.038 rad | 0.000 rad/s |
   | contactos OFF (ahora) | +5.740 rad | +0.944 rad/s |

   O sea que el brazo tenia 3 DOF utiles, no 4, en silencio. El URDF ahora trae un bloque `<mujoco>` que desactiva contactos, con la explicacion completa adentro. Colisionar cascos convexos entre piezas disenadas para tocarse no aporta nada util; si despues necesitas contacto real (piso, objeto agarrado), saca ese flag y pon primitivas de colision explicitas.

In [58]:
import mujoco as mj
import mujoco.viewer
from pathlib import Path
import time

In [59]:
# URDF hecho a mano (ya no esta en el arbol):
# URDF_RELATIVE_PATH = Path("urdf/freshfile_description/urdf/freshfile_mujoco.urdf")
# .gazebo no lo carga MuJoCo, es un fragmento de xacro para Gazebo:

XML_PATH = Path("New_folder/freshfile_mujoco.xml")  # export de Fusion2Mujoco
XML_PATH2 = Path("results_fusion2mujoco/meshes/freshfile_mujoco.xml")

def find_repo_root(start: Path, marker: Path) -> Path:
    """Sube desde `start` hasta encontrar un ancestro que contenga `marker`."""
    for candidate in [start, *start.parents]:
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(
        f"No se encontro '{marker}' subiendo desde {start}. "
        "Corre el notebook dentro del repo ERP, o ajusta XML_PATH."
    )

repo_root = find_repo_root(Path.cwd(), XML_PATH2)
model_path = (repo_root / XML_PATH2).resolve()
print(f"Cargando modelo MuJoCo desde: {model_path}")

model = mj.MjModel.from_xml_path(str(model_path))
data = mj.MjData(model)

print(f"Modelo cargado OK -> bodies: {model.nbody}, joints: {model.njnt}, "
      f"geoms: {model.ngeom}, actuadores: {model.nu}")
for j in range(model.njnt):
    print("  joint", j, mj.mj_id2name(model, mj.mjtObj.mjOBJ_JOINT, j))


Cargando modelo MuJoCo desde: C:\Users\nicoa\Desktop\Universidad\ERP\results_fusion2mujoco\meshes\freshfile_mujoco.xml
Modelo cargado OK -> bodies: 6, joints: 4, geoms: 5, actuadores: 4
  joint 0 rot
  joint 1 link1
  joint 2 link2
  joint 3 act


## Perillas del visor

`SLOWMO_ON` es el interruptor de camara lenta: en `False` el visor corre en tiempo real, en `True` reproduce `SLOWMO_FACTOR` veces mas lento. El ritmo se mantiene en los dos casos (el `sleep` sigue ahi con factor 1.0), asi que apagarlo no deja la sim corriendo a toda velocidad.

Los otros dos knobs existen porque las 4 juntas son `continuous`, sin limites: sin nada de amortiguamiento el brazo cae y gira para siempre — en 30 s de sim llega a picos de 54 rad/s y acumula ~70 rad. Con `JOINT_DAMPING = 0.05` se asienta y se queda quieto, que es lo que uno quiere mirar.

Ojo: ese amortiguamiento es **para poder ver**, no un valor identificado del hardware real. No lo uses como si fuera friccion medida.

El export ya trae 4 servos de posicion (`nu=4`), asi que en cuanto escribas `data.ctrl[:]` manda el servo y no la caida libre.

In [60]:
SLOWMO_ON = False      # <-- interruptor: False = tiempo real, True = camara lenta
SLOWMO_FACTOR = 10.0   # cuanto mas lento cuando SLOWMO_ON = True
JOINT_DAMPING = 0.05   # N*m*s/rad; 0.0 = caida libre eterna (ver nota arriba)
GRAVITY_ON = True      # False = el brazo se queda como esta, para inspeccionarlo

slowmo = SLOWMO_FACTOR if SLOWMO_ON else 1.0

model.dof_damping[:] = JOINT_DAMPING
model.opt.gravity[:] = (0, 0, -9.81) if GRAVITY_ON else (0, 0, 0)

# Con el factor alto cada paso duerme mas y el render se pone choppy: un paso de
# 2 ms a factor 10 son 20 ms de pared (50 fps), pero a factor 50 son 100 ms
# (10 fps). Si necesitas mas lento y fluido, baja tambien model.opt.timestep.
estado = f"ON (x{SLOWMO_FACTOR:g})" if SLOWMO_ON else "OFF (tiempo real)"
print(f"camara lenta: {estado}")
print(f"timestep={model.opt.timestep} s -> {model.opt.timestep * slowmo * 1e3:.0f} ms de pared por paso "
      f"(~{1 / (model.opt.timestep * slowmo):.0f} fps)")

camara lenta: OFF (tiempo real)
timestep=0.002 s -> 2 ms de pared por paso (~500 fps)


In [61]:
mj.mj_resetData(model, data)

with mujoco.viewer.launch_passive(model, data) as viewer:
    wall_start = time.time()
    while viewer.is_running():
        # --- tu codigo va aca, p.ej. data.ctrl[:] = ... ---
        mujoco.mj_step(model, data)
        # ---------------------------------------------------

        viewer.sync()

        # Ritmo de reproduccion (slowmo = 1.0 es tiempo real). Se compara contra
        # el tiempo de sim ABSOLUTO (no se suma paso a paso) para que un paso que
        # se pasa de largo no deje al visor corriendo adelantado para siempre.
        sleep_s = (wall_start + data.time * slowmo) - time.time()
        if sleep_s > 0:
            time.sleep(sleep_s)

print(f"Visor cerrado tras {data.time:.2f} s de simulacion.")

Visor cerrado tras 38.54 s de simulacion.
